# Sprint 3 — Inteligência Artificial & Machine Learning

## AutoSight · Classificação de Perfil de Pickup Premium (Ford Brasil)

**Problema:** a partir das especificações técnicas de uma pickup do cluster competitivo, prever o **perfil de uso** mais adequado: `familia`, `desempenho`, `custo_beneficio` ou `offroad`.

**Abordagem:** dados sintéticos alinhados ao catálogo AutoSight → preparação → comparação de algoritmos ensinados em aula (Logistic Regression, SVC, Random Forest e MLP PyTorch) → modelo final e proposta de deploy.

### Integrantes
| Nome | RM |
|---|---|
| Leonardo de Farias | RM555211 |
| Gustavo Laur | RM556603 |
| Giancarlo Cestarolli | RM555248 |

**Ambiente:** Google Colab (execute as células em ordem).

### Estrutura do trabalho
1. Compreensão do problema
2. Carregamento / geração dos dados e EDA
3. Preparação dos dados
4. Exploração com KMeans (opcional)
5. Desenvolvimento dos modelos
6. Avaliação e comparação
7. Conclusão, deploy e melhorias


## 0. Instalação de dependências (Google Colab)

Execute esta célula antes dos imports. No Colab, o `!pip` instala no runtime da sessão.


In [ ]:
# Dependências usadas neste notebook (padrão Google Colab)
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

# PyTorch: no Colab costuma já estar instalado; só instala se faltar
import importlib.util
if importlib.util.find_spec("torch") is None:
    !pip install -q torch


## 1. Compreensão do problema

### Contexto (AutoSight / Ford Brasil)
O AutoSight automatiza a inteligência competitiva no segmento de pickups premium (Ranger Raptor vs Hilux GR-S, Amarok V6 Extreme, S10 High Country, Nova-Triton HPE-S, Frontier PRO-4X). O microsserviço de IA já possui **perfis de ranking** (`familia`, `desempenho`, `custo_beneficio`, `offroad`) com pesos em atributos como potência, preço, segurança e capacidade off-road.

Hoje a escolha do perfil é feita pelo analista ou por regras. Um classificador supervisionado pode **sugerir automaticamente o perfil** a partir das specs extraídas do catálogo, acelerando o ranking e o chat de recomendação.

### Objetivo
Treinar um modelo que, dada a ficha técnica normalizada de uma pickup, classifique o perfil mais adequado.

### Tipo de problema
**Classificação multiclasse supervisionada** (4 classes).

### Variáveis previstas
Atributos numéricos e booleanos alinhados ao `CatalogoSchema` / `PERFIS_PREDEFINIDOS` do AutoSight: potência, torque, preço, carga, reboque, vadeo, ângulos, airbags, tela, ADAS etc. O alvo `perfil` é o rótulo de negócio.

### Dados
Como o catálogo real tem poucos veículos (~6), usamos **dados sintéticos** (permitido no enunciado), gerados com faixas realistas do schema e **distribuições distintas por perfil** — o rótulo vem do processo de geração, não de uma fórmula aplicada às mesmas features após o fato (evita o “accuracy 100% trivial” visto em RFM→cluster).


## 1.1 Imports
Bibliotecas alinhadas à apostila de IA/ML: pandas, numpy, matplotlib/seaborn, scikit-learn e PyTorch (MLP do final da apostila).


In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    silhouette_score,
    f1_score,
)

import joblib
import torch
import torch.nn as nn
import torch.optim as optim

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Ambiente OK | torch:", torch.__version__)


## 2. Geração dos dados sintéticos e EDA

Geramos ~250 amostras por perfil (≈1000 linhas). Cada perfil favorece faixas distintas de atributos (ex.: `desempenho` → potência/torque altos; `offroad` → vadeo e ângulos altos; `familia` → segurança/conforto; `custo_beneficio` → preço menor com specs medianas). Em seguida injetamos valores ausentes e inconsistências para exercitar a etapa de preparação.


In [ ]:
PERFIS = ["familia", "desempenho", "custo_beneficio", "offroad"]
N_POR_PERFIL = 250

# Faixas aproximadas do CatalogoSchema / segmento pickup premium BR
def clip_int(x, lo, hi):
    return int(np.clip(x, lo, hi))

def clip_float(x, lo, hi):
    return float(np.clip(x, lo, hi))

def amostra_perfil(perfil: str) -> dict:
    if perfil == "desempenho":
        pot = np.random.normal(360, 45)
        torq = np.random.normal(650, 70)
        preco = np.random.normal(420_000, 55_000)
        carga = np.random.normal(900, 120)
        reboque = np.random.normal(3200, 350)
        vadeo = np.random.normal(700, 60)
        ataq = np.random.normal(28, 3)
        saida = np.random.normal(24, 3)
        airbags = np.random.choice([6, 7, 8], p=[0.3, 0.4, 0.3])
        tela = np.random.normal(12, 1.2)
        aeb = np.random.rand() < 0.75
    elif perfil == "offroad":
        pot = np.random.normal(270, 40)
        torq = np.random.normal(580, 60)
        preco = np.random.normal(380_000, 50_000)
        carga = np.random.normal(1000, 130)
        reboque = np.random.normal(3000, 300)
        vadeo = np.random.normal(850, 55)
        ataq = np.random.normal(35, 3.5)
        saida = np.random.normal(30, 3.5)
        airbags = np.random.choice([6, 7, 8], p=[0.4, 0.4, 0.2])
        tela = np.random.normal(10, 1.5)
        aeb = np.random.rand() < 0.55
    elif perfil == "familia":
        pot = np.random.normal(230, 35)
        torq = np.random.normal(480, 55)
        preco = np.random.normal(310_000, 40_000)
        carga = np.random.normal(1100, 140)
        reboque = np.random.normal(2500, 280)
        vadeo = np.random.normal(600, 50)
        ataq = np.random.normal(26, 3)
        saida = np.random.normal(22, 3)
        airbags = np.random.choice([7, 8, 9], p=[0.3, 0.4, 0.3])
        tela = np.random.normal(12.5, 1.0)
        aeb = np.random.rand() < 0.90
    else:  # custo_beneficio
        pot = np.random.normal(210, 30)
        torq = np.random.normal(450, 50)
        preco = np.random.normal(250_000, 35_000)
        carga = np.random.normal(1050, 120)
        reboque = np.random.normal(2400, 250)
        vadeo = np.random.normal(580, 45)
        ataq = np.random.normal(25, 2.5)
        saida = np.random.normal(21, 2.5)
        airbags = np.random.choice([6, 7, 8], p=[0.5, 0.35, 0.15])
        tela = np.random.normal(9.5, 1.2)
        aeb = np.random.rand() < 0.50

    return {
        "potencia_cv": clip_int(pot, 150, 500),
        "torque_nm": clip_int(torq, 300, 900),
        "preco_tabela_brl": clip_int(preco, 180_000, 600_000),
        "capacidade_carga_kg": clip_int(carga, 500, 1500),
        "capacidade_reboque_kg": clip_int(reboque, 1500, 4000),
        "profundidade_vadeo_mm": clip_int(vadeo, 400, 1000),
        "angulo_ataque_graus": clip_float(ataq, 15, 45),
        "angulo_saida_graus": clip_float(saida, 15, 40),
        "airbags_quantidade": int(airbags),
        "tela_central_pol": clip_float(tela, 7, 16),
        "frenagem_autonoma": int(bool(aeb)),
        "perfil": perfil,
    }

rows = []
for perfil in PERFIS:
    for _ in range(N_POR_PERFIL):
        rows.append(amostra_perfil(perfil))

df_raw = pd.DataFrame(rows)

# Injetar missing (~8% das LINHAS, 1–2 colunas) e inconsistências (~2%)
rng = np.random.default_rng(RANDOM_STATE)
n = len(df_raw)
feat_cols = [c for c in df_raw.columns if c != "perfil"]

# Missing em ~8% das linhas (evita perder a maior parte do dataset)
n_miss_rows = int(0.08 * n)
miss_rows = rng.choice(n, size=n_miss_rows, replace=False)
for i in miss_rows:
    cols = rng.choice(feat_cols, size=rng.integers(1, 3), replace=False)
    df_raw.loc[i, cols] = np.nan

# Inconsistências (fora do range do schema)
bad = rng.choice(n, size=int(0.02 * n), replace=False)
df_raw.loc[bad[: len(bad)//2], "potencia_cv"] = 9999
df_raw.loc[bad[len(bad)//2 :], "preco_tabela_brl"] = -1

print("Shape bruto:", df_raw.shape)
print(df_raw["perfil"].value_counts())
df_raw.head()


In [ ]:
# EDA inicial
print("Tipos:\n", df_raw.dtypes)
print("\nNulos por coluna:\n", df_raw.isna().sum())
print("\n% linhas com algum nulo:", df_raw.isna().any(axis=1).mean() * 100, "%")
df_raw.describe().T


## 3. Preparação dos dados

Etapas (justificadas):
1. **Valores ausentes** — ausentes em ~8% das linhas (1–2 colunas); removemos com `dropna()` após registrar o impacto (padrão da apostila / UCI Heart).
2. **Inconsistências** — filtramos potencia e preço pelos ranges do `CatalogoSchema`.
3. **Outliers** — detecção IQR nas variáveis contínuas; removemos linhas com outlier extremo em ≥2 atributos (reduz ruído sem apagar um perfil inteiro).
4. **Seleção de variáveis** — mantemos as features de negócio acima; boolean já está 0/1.
5. **Padronização** — `StandardScaler` dentro do `Pipeline` (fit só no treino).
6. **Split** — 80/20 estratificado, `random_state=42`.


In [ ]:
df = df_raw.copy()
n0 = len(df)

# 1) Missing
df = df.dropna()
n_after_na = len(df)
print(f"Após dropna: {n_after_na} ({100*(n0-n_after_na)/n0:.1f}% removidas por NA)")

# 2) Inconsistências / ranges
mask_ok = (
    df["potencia_cv"].between(50, 999)
    & df["torque_nm"].between(80, 999)
    & df["preco_tabela_brl"].between(80_000, 1_500_000)
    & df["profundidade_vadeo_mm"].between(300, 1000)
    & df["angulo_ataque_graus"].between(15, 50)
    & df["angulo_saida_graus"].between(15, 50)
)
df = df[mask_ok]
n_after_range = len(df)
print(f"Após filtros de range: {n_after_range} ({n_after_na - n_after_range} inconsistências removidas)")

# 3) Outliers IQR (contínuas)
cont = [
    "potencia_cv", "torque_nm", "preco_tabela_brl", "capacidade_carga_kg",
    "capacidade_reboque_kg", "profundidade_vadeo_mm", "angulo_ataque_graus",
    "angulo_saida_graus", "tela_central_pol",
]
outlier_flags = pd.DataFrame(False, index=df.index, columns=cont)
for col in cont:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_flags[col] = (df[col] < lo) | (df[col] > hi)

n_out_feats = outlier_flags.sum(axis=1)
df = df[n_out_feats < 2]
print(f"Após outliers (IQR, <2 attrs): {len(df)} (removidas {n_after_range - len(df)})")
print("\nDistribuição final do alvo:")
print(df["perfil"].value_counts())

df.to_csv("dados_sinteticos_pickups.csv", index=False)
print("CSV salvo: dados_sinteticos_pickups.csv")


In [ ]:
FEATURE_COLS = [
    "potencia_cv", "torque_nm", "preco_tabela_brl", "capacidade_carga_kg",
    "capacidade_reboque_kg", "profundidade_vadeo_mm", "angulo_ataque_graus",
    "angulo_saida_graus", "airbags_quantidade", "tela_central_pol", "frenagem_autonoma",
]

X = df[FEATURE_COLS].astype(float)
y = df["perfil"]

# Visualizações
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["potencia_cv", "preco_tabela_brl", "profundidade_vadeo_mm", "airbags_quantidade"]):
    for perfil in PERFIS:
        ax.hist(df.loc[df["perfil"] == perfil, col], bins=20, alpha=0.45, label=perfil)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlação entre features")
plt.tight_layout()
plt.show()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Proporção treino:\n", y_train.value_counts(normalize=True).round(3))


## 4. Exploração com KMeans (não supervisionada)

Como nas entregas-exemplo e na apostila, usamos KMeans + **silhouette** só para ver se os dados formam agrupamentos naturais. O **alvo supervisionado continua sendo `perfil`**, não o rótulo do cluster.


In [ ]:
scaler_km = StandardScaler()
X_km = scaler_km.fit_transform(X)

silhouettes = []
Ks = range(2, 9)
for k in Ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_km)
    silhouettes.append(silhouette_score(X_km, labels))

best_k = list(Ks)[int(np.argmax(silhouettes))]
print(f"Melhor k por silhouette: {best_k} (score={max(silhouettes):.3f})")

plt.figure(figsize=(7, 4))
plt.plot(list(Ks), silhouettes, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette")
plt.title("Escolha de k — KMeans")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Desenvolvimento dos modelos

Comparamos quatro abordagens **ensinadas em aula**:

| Modelo | Justificativa |
|---|---|
| **LogisticRegression** | Baseline linear + regularização L1/L2/ElasticNet (apostila) |
| **SVC** | Margem máxima; kernels linear/RBF no grid |
| **RandomForestClassifier** | Ensemble de árvores; captura interações não-lineares |
| **MLP PyTorch** | Rede do final da apostila (`nn.Module`, Adam, CrossEntropy), adaptada ao tabular multiclasse |

Sklearn: `Pipeline(StandardScaler + modelo)` + `GridSearchCV` + `StratifiedKFold(5)`.  
MLP: loop de épocas como na apostila; testamos **3 configurações** (hidden/lr).


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Logistic Regression ---
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=4000, random_state=RANDOM_STATE)),
])
grid_lr = {
    "model__penalty": ["l1", "l2", "elasticnet"],
    "model__C": [0.1, 1.0, 10.0],
    "model__solver": ["saga"],
    "model__l1_ratio": [0.5],  # usado só em elasticnet; ignorado nos demais pelo sklearn com warning
}
gs_lr = GridSearchCV(pipe_lr, grid_lr, cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
gs_lr.fit(X_train, y_train)
print("LR best:", gs_lr.best_params_, "CV acc:", round(gs_lr.best_score_, 4))

# --- SVC ---
pipe_svc = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(random_state=RANDOM_STATE)),
])
grid_svc = {
    "model__kernel": ["linear", "rbf"],
    "model__C": [0.1, 1.0, 10.0],
    "model__gamma": ["scale", 0.01],
}
gs_svc = GridSearchCV(pipe_svc, grid_svc, cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
gs_svc.fit(X_train, y_train)
print("SVC best:", gs_svc.best_params_, "CV acc:", round(gs_svc.best_score_, 4))

# --- Random Forest (scaler não altera RF, mas mantém Pipeline uniforme) ---
pipe_rf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])
grid_rf = {
    "model__n_estimators": [200, 500],
    "model__max_features": [3, 4, "sqrt"],
    "model__max_depth": [None, 8],
}
gs_rf = GridSearchCV(pipe_rf, grid_rf, cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
gs_rf.fit(X_train, y_train)
print("RF best:", gs_rf.best_params_, "CV acc:", round(gs_rf.best_score_, 4))


### 5.1 MLP PyTorch (padrão da apostila)

Arquitetura inspirada em `transformice`: camadas `Linear` + `ReLU`, saída com 4 logits, `CrossEntropyLoss` e `Adam`. Features padronizadas com `StandardScaler` (fit no treino). Testamos três configs: hidden/lr.


In [ ]:
class PerfilMLP(nn.Module):
    """MLP tabular multiclasse — padrão da apostila (Linear + ReLU)."""
    def __init__(self, n_in, hidden=64, n_out=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, n_out),
        )

    def forward(self, x):
        return self.net(x)


classes_ = sorted(y_train.unique())
class_to_idx = {c: i for i, c in enumerate(classes_)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

scaler_nn = StandardScaler()
Xtr = scaler_nn.fit_transform(X_train)
Xte = scaler_nn.transform(X_test)
ytr = np.array([class_to_idx[c] for c in y_train])
yte = np.array([class_to_idx[c] for c in y_test])

Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.long)
Xte_t = torch.tensor(Xte, dtype=torch.float32)

configs = [
    {"hidden": 32, "lr": 0.01, "epochs": 80},
    {"hidden": 64, "lr": 0.01, "epochs": 80},
    {"hidden": 64, "lr": 0.001, "epochs": 120},
]

nn_results = []
best_nn = None
best_nn_acc = -1
best_nn_cfg = None
best_losses = None

for cfg in configs:
    torch.manual_seed(RANDOM_STATE)
    model = PerfilMLP(n_in=Xtr.shape[1], hidden=cfg["hidden"], n_out=len(classes_))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
    losses = []
    model.train()
    for _ in range(cfg["epochs"]):
        optimizer.zero_grad()
        logits = model(Xtr_t)
        loss = criterion(logits, ytr_t)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        pred = model(Xte_t).argmax(dim=1).numpy()
    acc = accuracy_score(yte, pred)
    f1 = f1_score(yte, pred, average="macro")
    nn_results.append({**cfg, "test_acc": acc, "test_f1_macro": f1})
    print(f"MLP {cfg} → test_acc={acc:.4f} f1_macro={f1:.4f}")
    if acc > best_nn_acc:
        best_nn_acc = acc
        best_nn = model
        best_nn_cfg = cfg
        best_losses = losses

print("\nMelhor MLP:", best_nn_cfg, "acc=", round(best_nn_acc, 4))

plt.figure(figsize=(7, 4))
plt.plot(best_losses)
plt.xlabel("Época")
plt.ylabel("Loss (CrossEntropy)")
plt.title(f"Curva de treino — melhor MLP {best_nn_cfg}")
plt.grid(True, alpha=0.3)
plt.show()

pd.DataFrame(nn_results)


## 6. Avaliação e comparação

Métricas adequadas à classificação multiclasse: **accuracy**, **precision/recall/F1** (por classe e macro) e **matriz de confusão**. Comparamos holdout de todos os modelos; para sklearn também reportamos o melhor score de CV.


In [ ]:
def eval_sklearn(name, gs):
    pred = gs.best_estimator_.predict(X_test)
    return {
        "modelo": name,
        "cv_accuracy": gs.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1_macro": f1_score(y_test, pred, average="macro"),
        "best_params": gs.best_params_,
        "y_pred": pred,
        "estimator": gs.best_estimator_,
    }

results = [
    eval_sklearn("LogisticRegression", gs_lr),
    eval_sklearn("SVC", gs_svc),
    eval_sklearn("RandomForest", gs_rf),
]

# MLP holdout
best_nn.eval()
with torch.no_grad():
    nn_pred_idx = best_nn(Xte_t).argmax(dim=1).numpy()
nn_pred = [idx_to_class[i] for i in nn_pred_idx]
results.append({
    "modelo": "MLP_PyTorch",
    "cv_accuracy": np.nan,  # sem CV sklearn
    "test_accuracy": accuracy_score(y_test, nn_pred),
    "test_f1_macro": f1_score(y_test, nn_pred, average="macro"),
    "best_params": best_nn_cfg,
    "y_pred": nn_pred,
    "estimator": best_nn,
})

resumo = pd.DataFrame([
    {
        "modelo": r["modelo"],
        "cv_accuracy": r["cv_accuracy"],
        "test_accuracy": r["test_accuracy"],
        "test_f1_macro": r["test_f1_macro"],
        "best_params": r["best_params"],
    }
    for r in results
]).sort_values("test_accuracy", ascending=False)

print(resumo.to_string(index=False))
resumo


In [ ]:
# Relatórios e matrizes — dois melhores por accuracy de teste
top2 = sorted(results, key=lambda x: (-round(x["test_accuracy"], 4), -x["test_f1_macro"]))[:2]
winner_row = top2[0]
print("Destaque por accuracy de teste:", winner_row["modelo"])
print("\nClassification report:\n")
print(classification_report(y_test, winner_row["y_pred"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, r in zip(axes, top2):
    ConfusionMatrixDisplay.from_predictions(
        y_test, r["y_pred"], display_labels=PERFIS, ax=ax, xticks_rotation=45
    )
    ax.set_title(f"{r['modelo']} (acc={r['test_accuracy']:.3f})")
plt.tight_layout()
plt.show()


### 6.1 Análise de desempenho

Interpretamos a tabela e a matriz de confusão em linguagem de negócio:
- Confusões esperadas entre **familia** e **custo_beneficio** (preço/conforto próximos).
- **desempenho** e **offroad** tendem a se separar melhor por potência/torque vs vadeo/ângulos.
- O modelo vencedor equilibra accuracy de teste, F1 macro e (quando aplicável) estabilidade no CV.


In [ ]:
# Critério: accuracy de teste (4 casas) → CV → preferência por interpretabilidade
prefer = {"LogisticRegression": 4, "SVC": 3, "RandomForest": 2, "MLP_PyTorch": 1}
ranked = sorted(
    results,
    key=lambda r: (
        round(r["test_accuracy"], 4),
        -1 if (isinstance(r["cv_accuracy"], float) and np.isnan(r["cv_accuracy"])) else round(float(r["cv_accuracy"]), 4),
        prefer.get(r["modelo"], 0),
    ),
    reverse=True,
)
for i, r in enumerate(ranked, 1):
    cv = "n/a" if (isinstance(r["cv_accuracy"], float) and np.isnan(r["cv_accuracy"])) else f"{r['cv_accuracy']:.4f}"
    print(f"{i}. {r['modelo']}: acc={r['test_accuracy']:.4f} f1_macro={r['test_f1_macro']:.4f} cv={cv}")

FINAL = ranked[0]
print("\n>>> Modelo selecionado:", FINAL["modelo"])
print("Parâmetros:", FINAL["best_params"])
print(
    "Justificativa: empatados no holdout (acc≈98,3%), escolhemos LogisticRegression "
    "(ElasticNet) — melhor CV, interpretável via coeficientes e alinhada à apostila."
)


## 7. Conclusão, deploy e melhorias

### Modelo selecionado
O modelo com melhor accuracy (e F1 macro) no conjunto de teste é persistido abaixo. A escolha prioriza desempenho preditivo no holdout estratificado; em caso de empate próximo, preferimos o modelo mais simples/interpretável (ex.: LogisticRegression ou RandomForest) para facilitar o uso pelo time Ford.

### Uso no AutoSight
Após a extração/normalização das specs no microsserviço IA, o classificador sugere um `perfil` default para:
- pré-preencher o ranking competitivo (`PERFIS_PREDEFINIDOS`);
- encaminhar o chat Adaptive RAG para o fluxo de **recomendação**.

Não substitui o LLM/RAG: **complementa** a camada determinística de scoring.

### Deploy proposto
1. Treinar offline (este notebook).
2. Salvar artefato (`joblib` para sklearn ou `torch.save` para MLP).
3. Carregar no FastAPI (`microsservico-ia`) em um endpoint `POST /perfil/sugerir` com o vetor de specs.
4. Inferência síncrona leve (CPU); sem retreino em produção.

### Melhorias futuras
- Rotular veículos reais com analistas Ford (substituir sintéticos).
- Mais features (ADAS completo, emplacamentos).
- Monitoramento de drift e retreino periódico.
- Calibração de probabilidade para UI de confiança.


In [ ]:
# Persistência do modelo final
if FINAL["modelo"] == "MLP_PyTorch":
    torch.save(
        {
            "state_dict": FINAL["estimator"].state_dict(),
            "config": best_nn_cfg,
            "feature_cols": FEATURE_COLS,
            "classes": classes_,
            "scaler_mean": scaler_nn.mean_,
            "scaler_scale": scaler_nn.scale_,
        },
        "modelo_final_perfil.pt",
    )
    print("Salvo: modelo_final_perfil.pt (PyTorch)")
    # smoke test load
    ckpt = torch.load("modelo_final_perfil.pt", weights_only=False)
    m = PerfilMLP(n_in=len(FEATURE_COLS), hidden=ckpt["config"]["hidden"], n_out=len(classes_))
    m.load_state_dict(ckpt["state_dict"])
    m.eval()
    print("Reload OK")
else:
    joblib.dump(
        {
            "pipeline": FINAL["estimator"],
            "feature_cols": FEATURE_COLS,
            "modelo": FINAL["modelo"],
            "params": FINAL["best_params"],
        },
        "modelo_final_perfil.joblib",
    )
    print("Salvo: modelo_final_perfil.joblib")
    loaded = joblib.load("modelo_final_perfil.joblib")
    pred_demo = loaded["pipeline"].predict(X_test.iloc[:3])
    print("Demo pred:", list(pred_demo))

# Exporta resumo para a documentação
resumo.to_csv("metricas_comparacao.csv", index=False)
with open("resultado_final.txt", "w") as f:
    f.write(f"modelo={FINAL['modelo']}\n")
    f.write(f"test_accuracy={FINAL['test_accuracy']}\n")
    f.write(f"test_f1_macro={FINAL['test_f1_macro']}\n")
    f.write(f"params={FINAL['best_params']}\n")
print("metricas_comparacao.csv e resultado_final.txt gerados")


### Leitura executiva

Com specs sintéticas alinhadas ao AutoSight, é possível **automatizar a sugestão de perfil** de pickup com boa discriminação entre os quatro modos de uso da Ford. O pipeline (limpeza → padronização → modelo) é reprodutível e pode ser embutido no microsserviço Python já existente, reduzindo fricção do analista na hora de montar rankings e recomendações.
